## ECC implementation and visualization

Gerard Consuelo, Atay Moya, Larry Shields

### Overview

RSA is the most widely accepted public-key scheme. It is used for encryption, digital signatures, and symmetric key exchange. Though quite powerful in that it is computationally impossible to reverse its process, it is very computational heavy. Its key sizes are enormous at around 512 bits, and must keep increasing with the rapid development of technology. 

Elliptical Curve Cryptography (ECC) is a method of cryptography based on elliptical curves and the difficulty in computing the elliptic curve discrete logarithmic problem. It offers the same security as RSA, with the differnece that it is extremely efficient for smaller bit sizes, where a 256 -bit key for ECC provides as much security as a 3072-bit key using RSA.


### The Elliptic Curve Discrete Logarithmic Problem

In RSA, recall that the core equation we have was starting with two large prime numbers, p and q, to get our modulo. 
Then we would use Euler's Totient function (p-1)(q-1) to narrow it down to a finite amount of numbers. Then, we pick a public key e in between 1 and the totient. To achieve the private key d, we would find the public key's inverse through the modulo stuff. To encrypt, the ciphertext (c) is m^e and to decrypt, the plaintext (m) should be the result from c^d.

ECC follows a similar process, but with points on the elliptic curve.
The core equation we start with for ECC is k * P = Q, which is the elliptic curve discrete logarithmic problem. 
In this equation, k is a scalar, and P and Q are points on a given curve. 
For encryption, k is our private key, Q is our public key, and P is a given constant starting point.


### The Algebra

We first have to look at the algebra of an elliptic curve. 
The key operations we need to examine are point addition and point doubling. 

**Point Addition**
This operation is defined as $R = P + Q$, where $P$, $Q$, and the resulting point $R$ all lie on the curve, or are the identity element, denoted as $\mathcal{O}$ (often referred to as the Point at Infinity). 

Elliptic curves have a geometric property where drawing a straight line through any 2 points ($P$ and $Q$) will intersect the curve at exactly one third point, which we call $-R$. To find the actual result of $P + Q$, you take that third intersection point and **reflect it across the x-axis** to get $R$. 

An edge case occurs when $Q = -P$ (meaning they share the same x-coordinate but have opposite y-coordinates). The line through them is purely vertical, meaning it never intersects the curve a third time. Instead, we say it intersects at infinity, resulting in our identity value: $P + (-P) = \mathcal{O}$. 

Assuming our curve follows the standard form $y^2 = x^3 + ax + b$, and $P = (x_1, y_1)$ and $Q = (x_2, y_2)$, here are the equations to find $R = (x_3, y_3)$:

$$λ = \frac{y_2 - y_1}{x_2 - x_1}$$
$$x_3 = λ^2 - x_1 - x_2$$
$$y_3 = λ(x_1 - x_3) - y_1$$

**Point Doubling**
Building on top of addition, this is simply calculating $2P = P + P$. Geometrically, since $P$ and $Q$ are the exact same point, you cannot draw a standard line between them. Instead, you draw the **tangent line** to the curve exactly at point $P$. This tangent line will intersect the curve at another point, $-2P$. Just like in addition, you reflect this point across the x-axis to find $2P$.

$2P$ results in the identity value $\mathcal{O}$ if $P$ is already the identity value, or if the y-coordinate of $P$ is exactly $0$ (which makes the tangent line perfectly vertical).

Using the same curve parameters, here are the equations for point doubling to find $2P = (x_3, y_3)$:

$$λ = \frac{3x_1^2 + a}{2y_1}$$
$$x_3 = λ^2 - 2x_1$$
$$y_3 = λ(x_1 - x_3) - y_1$$

With these algebraic pieces, we now know the mechanics behind scalar multiplication, like $k \cdot P$ on an elliptic curve. This means adding $P$ to itself $k$ times:
$$P + P + P + \dots \text{ (k times)}$$

### Finite Field
However, the numbers can get way too large, leading to round off errors and sluggishness. To remedy this, we will use a modulo, allowing an infinite number of inputs but a finite number of outputs. There will be overlaps, making it much harder to reverse. 

So the equations will look like this:

**Point Addition ($R = P + Q$)**

points $P = (x_1, y_1)$ and $Q = (x_2, y_2)$:

$$\lambda \equiv (y_2 - y_1) \cdot (x_2 - x_1)^{-1} \pmod M$$
$$x_3 \equiv \lambda^2 - x_1 - x_2 \pmod M$$
$$y_3 \equiv \lambda(x_1 - x_3) - y_1 \pmod M$$

**Point Doubling ($2P$)**

point $P = (x_1, y_1)$ (where $y_1 eq 0$):

$$\lambda \equiv (3x_1^2 + a) \cdot (2y_1)^{-1} \pmod M$$
$$x_3 \equiv \lambda^2 - 2x_1 \pmod M$$
$$y_3 \equiv \lambda(x_1 - x_3) - y_1 \pmod M$$

### Modular Arithmetic

Because now we are in a finite field, the algebra changes as well. Here are the ones we are using above and the explanation. Notice there is no division, only multiplication. For these operations, we assume that the inputs are all within the modulo already. 

**Addition**
When adding two numbers in a finite field, the worst-case bit length increases by 1 (e.g., an 8-bit number might overflow into a 9-bit number). To keep the result within the field, we simply subtract the modulo M from the addition. 
* **Example:** $15 + 20 = 35$. Since $35 > 23$, we do $35 - 23 = 12$.

**Subtraction**
Similar to addition, subtraction worst case brings the bit length down or the value to become negative. To keep the result within the field, we simply add the modulo M to the result.
* **Example:** $15 - 20 = -5$. Since $-5 < 0$, we do $-5 + 23 = 18$.

**Multiplication (Double-and-Add Algorithm)**

Instead of multiplying two large numbers directly which can cause the bit length to overflow, we can rely on bitwise properties using the "double-and-add" method. We follow these steps.
1.  Get the number we multiply by, m, and get its bit representation.
2.  Loop through its bits
3.  if the bit is "on", then we add m to a separate value.
4.  double m (since every shift is *2 basically)





**Division using The Extended Euclidean Algorithm**

You cannot simply divide numbers in this finite space.  We must find A's modular inverse. Finding this inverse requires the Extended Euclidean Algorithm (EEA).
* Extended Euclidean Algorithm (EEA) involves Euclid's own algorithm to find the greatest common divisor (GCD) of two numbers.
  * Bézout finds that you can multiply a number m to one number and a number n to another and still get the same GCD.
  $$B \cdot m + P \cdot n = {GCD}(B, P)$$
  * Because $P$ is a prime number, the GCD of $B$ and $P$ is guaranteed to be exactly 1 as long as B is not 0. So, Bézout's identity simplifies perfectly to:
  $$B \cdot m + P \cdot n = 1$$
  * Now, if we take the modulo $P$ of both sides of this equation, $P \cdot n$ gets canceled out entirely because it is a direct multiple of the modulo $P$. We are left with:
  $$B \cdot m \equiv 1 \pmod P$$
  * Therefore, $m$ is our inverse! Once we use the EEA to find $m$, division simply becomes $A 	imes m \pmod P$.